In [1]:

import asyncio
import warnings
import pandas as pd
from pathlib import Path

# Парсеры
from antimony import antimony_parser
from westmetall import westmetall_async
from lme import lme_selenium_async
from lbma import lbma_prescious_async

from cbr import cb_currency, cb_metalls
from nbk import nbk_tenge_async
from shmet import shmet_optimized_async
from kitco import kitco_parser_async

# Сервисные функции из service_layer
from service_layer import (
    read_db,
    save_db,
    check_df,
    check_and_save_pair,
    show_db,
    excel_to_csv_db
)

warnings.filterwarnings("ignore")


# ================================================================
# Пути к базам (реальная структура проекта)
# ================================================================
LME_PATH = Path("lme/data/LME_db_new.xlsx")
WESTMETALL_PATH = Path("westmetall/data/LME_westmetall_db.xlsx")

KITCO_PATH = Path("kitco/data/kitko_db.xlsx")
LBMA_PATH = Path("lbma/data/lbma_kitco_subs.xlsx")

ANTIMONY_PATH = Path("antimony/data/antimony.xlsx")

CB_CURRENCY_PATH = Path("cbr/data/cb_currency.xlsx")
CB_METALLS_PATH = Path("cbr/data/cb_metalls.xlsx")

NBK_PATH = Path("nbk/data/nbk_tenge.xlsx")
SHMET_PATH = Path("shmet/data/shmet_historical.xlsx")


# ================================================================
# Проверка целостности парных баз
# ================================================================
def db_check():
    """
    Проверка целостности парных баз с очисткой дубликатов
    и сохранением первого (более раннего) вхождения.
    """
    print("Проверка LME / Westmetall...")
    check_and_save_pair(
        LME_PATH,
        WESTMETALL_PATH,
        pair_name="LME / Westmetall",
        index=False,
    )

    print("Проверка Kitco / LBMA...")
    check_and_save_pair(
        KITCO_PATH,
        LBMA_PATH,
        pair_name="Kitco / LBMA",
        index=False,
    )
    
async def main():
    print("Parsing started...")

    tasks = {
        "lme": lme_selenium_async(),
        "antimony": antimony_parser(),
        "westmetall": westmetall_async(),
        "lbma": lbma_prescious_async(),
        "kitco": kitco_parser_async(),
        "cb_currency": cb_currency(),
        "cb_metalls": cb_metalls(),
        "nbk": nbk_tenge_async(),
        "shmet": shmet_optimized_async(),
    }

    # return_exceptions=True — чтобы падение одного парсера
    # не останавливало остальные
    results = await asyncio.gather(
        *tasks.values(),
        return_exceptions=True,
    )

    for name, result in zip(tasks.keys(), results):
        if isinstance(result, Exception):
            print(f"❌ Ошибка в {name}: {result}")

    print("All tasks are done!")
    print("+" * 64)
    print("Checking DB...")

    db_check()

    print("DB check completed!")
    print("+" * 64)
    print("Visual control")
    print("+" * 64)
    
    print("Converting Excel to CSV...")
    excel_to_csv_db()
    print("CSV conversion completed!")

    # Базовые металлы
    show_db("lme_selenium_db", LME_PATH, sheet_name=0)
    show_db("westmetall_db", WESTMETALL_PATH, sheet_name=0)

    # Драгоценные металлы
    show_db("kitco_db", KITCO_PATH, sheet_name=0)
    show_db("lbma_precious_db", LBMA_PATH, sheet_name=0)

    # Антимоний
    show_db("antimony_db", ANTIMONY_PATH, sheet_name=0)

    # ЦБ РФ: валюты (каждая на своем листе)
    for currency in [
        "USD",
        "EUR",
        "British_Pound",
        "China_Yuan",
        "Japanese_Yen",
        "Swiss_Franc",
    ]:
        show_db(
            f"cb_currency ({currency})",
            CB_CURRENCY_PATH,
            sheet_name=currency,
        )

    # ЦБ РФ: металлы
    show_db("cb_metalls_db", CB_METALLS_PATH, sheet_name=0)

    # Казахстан и SHMET
    show_db("nbk_tenge_db", NBK_PATH, sheet_name=0)
    show_db("shmet_historical_db", SHMET_PATH, sheet_name=0, show_head=True)


# Для Jupyter используем await, а не asyncio.run()
await main()

Parsing started...
🚀 LME parsing started...
antimony parsing is DONE
NBK_tenge parsing is DONE! (1739 строк)
✅ LME_main is done!!!
CB_metalls parsing is DONE!
USD is done!
EUR is done!
Australian_Dollar is done!
China_Yuan is done!
British_Pound is done!
Kazakhstan_Tenge is done!
Japanese_Yen is done!
Swiss_Franc is done!
CB_currency parsing is DONE!
WESTMETALL is done!!!
Произошла ошибка KITCO: Не найдены блоки <div class='grid'> на странице Kitco
SHMET is done!!!
[LBMA/Selenium] Открываем страницу LBMA...
[LBMA/Selenium] Обрабатываем Gold...
[LBMA/Selenium] Сетевых ответов от prices.lbma.org.uk: 1
[LBMA/Selenium] Gold: получено 10 строк
[LBMA/Selenium] Обрабатываем Silver...
[LBMA/Selenium] Сетевых ответов от prices.lbma.org.uk: 1
[LBMA/Selenium] Silver: получено 10 строк
[LBMA/Selenium] Обрабатываем Platinum...
[LBMA/Selenium] Сетевых ответов от prices.lbma.org.uk: 2
[LBMA/Selenium] Platinum: получено 10 строк
[LBMA/Selenium] Обрабатываем Palladium...
[LBMA/Selenium] Сетевых ответов

,date,lead,nickel,zink,tin,aluminium,copper
1220,2026-10-30,1890.0,15240,3955.5,48795,3114.0,14449.0
1221,2026-11-02,1869.5,15260,3935.0,49520,3095.0,14380.0
1222,2026-11-03,1862.0,15200,3869.0,47905,3082.0,14110.5
1223,2026-11-04,1858.0,15465,3875.0,48125,3135.0,14201.0
1224,2026-11-05,1866.5,16100,3877.0,48100,3160.0,14110.0


westmetall_db


,date,aluminium,copper,lead,nickel,zink,tin
1220,2026-10-30,3114.0,14449.0,1890.0,15240,3955.5,48795
1221,2026-11-02,3095.0,14380.0,1869.5,15260,3935.0,49520
1222,2026-11-03,3082.0,14110.5,1862.0,15200,3869.0,47905
1223,2026-11-04,3135.0,14201.0,1858.0,15465,3875.0,48125
1224,2026-11-05,3160.0,14110.0,1866.5,16100,3877.0,48100


kitco_db


,Date,Gold,Silver,Platinum,Palladium
14881,2026-10-30,3913.25,50.100,1426.95,974.85
14882,2026-11-02,3955.25,50.125,1443.35,973.30
14883,2026-11-03,3936.65,49.005,1447.45,946.00
14884,2026-11-04,4037.80,51.990,1468.70,988.25
14885,2026-11-05,4112.85,52.820,1530.45,1052.15


... (18 outputs left out)

nbk_tenge_db


,date,Числовое значение,ДОЛЛАР США
1734,2026-11-02,1,453.87
1735,2026-11-03,1,453.64
1736,2026-11-04,1,453.26
1737,2026-11-05,1,452.91
1738,2026-11-06,1,451.87


shmet_historical_db


,date,price,unit
0,2020-01-10,48605,Yuan/MT
1,2020-01-14,48990,Yuan/MT
2,2020-01-15,49060,Yuan/MT
3,2020-01-16,48950,Yuan/MT
4,2020-01-17,48930,Yuan/MT
